# Lab 14: Cross-Validation, Bootstrap & Model Selection
> Week 14 | CLO4 | ISLP Ch.5

## บทนำสัปดาห์

สัปดาห์นี้เราจะตอบคำถามสำคัญที่สุดใน Machine Learning: **"เราจะรู้ได้อย่างไรว่า model ไหนดีที่สุด?"** ปัญหาหลักคือ Training Error ต่ำไม่ได้แปลว่า Test Error ต่ำ — เราอาจ overfit ไปกับ training data โดยไม่รู้ตัว **Cross-Validation (CV)** แก้ปัญหานี้โดยจำลอง test set จาก training data เอง ทำให้เราประเมิน Test Error ได้โดยไม่ต้อง waste data ส่วน **Bootstrap** เป็นเทคนิค resampling ที่ powerful ที่สุด ใช้ประเมิน Standard Error ของ statistic ใดๆ ที่ไม่มีสูตรปิด **เป้าหมาย** คือนักศึกษาสามารถ: ใช้ k-Fold CV เลือก polynomial degree และ K ของ KNN ได้, ใช้ Bootstrap ประเมิน SE ของ regression coefficient ได้, และเข้าใจ connection กับ Bias-Variance trade-off จาก Week 6 ทักษะเหล่านี้เป็น foundation ของ Machine Learning engineering ที่ใช้ในทุก model ทุกระดับ

## สิ่งที่จะเรียนรู้
- Validation Set approach และข้อจำกัด
- LOOCV: low bias แต่ computationally expensive
- k-Fold CV: practical trade-off, K=5 หรือ K=10
- Bootstrap: SE estimation สำหรับ statistic ที่ซับซ้อน
- CV-based model selection: polynomial degree, KNN K

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────────────────
# วัตถุประสงค์: โหลด library สำหรับ Cross-Validation และ Bootstrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# ─── CV tools ──────────────────────────────────────────────────────────────────
# วัตถุประสงค์: import Cross-Validation tools จาก sklearn
from sklearn.model_selection import (
    train_test_split, KFold, LeaveOneOut,
    cross_val_score, cross_validate
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from scipy.special import expit

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Setup complete ✓')

## Part 1: Validation Set Approach และข้อจำกัด

**Part นี้เราจะสำรวจ Validation Set approach** ซึ่งเป็นวิธีที่ง่ายที่สุดในการประเมิน test error แนวคิดคือ split data ออกเป็น train/validation เพียงครั้งเดียว แล้ว report error บน validation set อย่างไรก็ตาม วิธีนี้มีปัญหาสำคัญ: ผลลัพธ์ขึ้นกับ random split มาก ทำให้ variance ของ estimated test error สูง เราจะสังเกตปัญหานี้ผ่าน simulation

In [ ]:
# ─── สร้าง Auto-like dataset ────────────────────────────────────────────────────
# วัตถุประสงค์: simulate Auto dataset สำหรับ polynomial regression CV
# ความสัมพันธ์จริง: mpg = f(horsepower) แบบ non-linear
np.random.seed(42)
n = 392
hp = np.random.uniform(50, 230, n)  # horsepower

# True relationship: quadratic (mpg ลดลงตาม hp แบบ curve)
mpg_true = 60 - 0.46*hp + 0.0015*hp**2
mpg = mpg_true + np.random.normal(0, 3, n)  # เพิ่ม noise

auto = pd.DataFrame({'mpg': mpg, 'horsepower': hp})
print(f'Auto dataset: {auto.shape}')
print(auto.describe().round(2))

# Plot ข้อมูล
plt.figure(figsize=(8, 4))
plt.scatter(auto['horsepower'], auto['mpg'], alpha=0.3, s=15, color='steelblue')
plt.xlabel('Horsepower'); plt.ylabel('MPG')
plt.title('Auto Dataset: MPG vs Horsepower')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Validation Set: Repeat 10 times แสดง variance ────────────────────────────
# วัตถุประสงค์: แสดงว่า validation set MSE ผันผวนมากตาม random split
# นี่คือ motivation ว่าทำไม CV ถึงดีกว่า single split

X_auto = auto[['horsepower']].values
y_auto = auto['mpg'].values

degrees = range(1, 6)
n_trials = 10
val_mse_all = np.zeros((n_trials, len(degrees)))

for trial in range(n_trials):
    # Random split ด้วย seed ต่างกันทุกรอบ
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_auto, y_auto, test_size=0.5, random_state=trial)
    
    for j, deg in enumerate(degrees):
        # Polynomial features
        poly = PolynomialFeatures(degree=deg, include_bias=False)
        lr   = LinearRegression()
        X_tr_poly = poly.fit_transform(X_tr)
        X_va_poly = poly.transform(X_va)
        lr.fit(X_tr_poly, y_tr)
        y_pred = lr.predict(X_va_poly)
        val_mse_all[trial, j] = mean_squared_error(y_va, y_pred)

# Plot: MSE vs degree สำหรับทุก trial
plt.figure(figsize=(9, 5))
for trial in range(n_trials):
    plt.plot(list(degrees), val_mse_all[trial], 'o-', alpha=0.4,
             color='steelblue', lw=1.5, markersize=4)
plt.xlabel('Polynomial Degree'); plt.ylabel('Validation MSE')
plt.title('Validation Set MSE — 10 Different Random Splits\n'
          '(สังเกตความแปรปรวนสูงของ MSE ระหว่าง splits)')
plt.tight_layout(); plt.show()
print('ปัญหา: MSE ผันผวนมากระหว่าง splits → ต้องการวิธีที่ stable กว่า!')

### 🔰 TODO 1 (Easy): LOOCV บน Auto Dataset

Leave-One-Out Cross-Validation (LOOCV) แก้ปัญหาความแปรปรวนโดยใช้ทุก observation เป็น validation ทีละ 1 ครั้ง ผลลัพธ์ deterministic (ไม่ขึ้นกับ random seed) และมี bias ต่ำ อย่างไรก็ตาม ต้อง fit model n ครั้ง ซึ่ง expensive สำหรับ n ใหญ่ ใน TODO นี้คุณจะใช้ sklearn `LeaveOneOut` เพื่อ compute LOOCV MSE สำหรับ polynomial degree 1–5

**สิ่งที่ต้องทำ:**
1. ใช้ `LeaveOneOut()` จาก sklearn + `cross_val_score()` สำหรับแต่ละ polynomial degree
2. คำนวณ LOOCV MSE = mean ของ cross_val_score (-MSE)
3. Plot LOOCV MSE vs polynomial degree
4. ระบุ optimal degree (minimum LOOCV MSE)

**Expected**: optimal degree ≈ 2 (เพราะ true relationship เป็น quadratic)

**Hint**: `cross_val_score(..., scoring='neg_mean_squared_error')` → ผลเป็น negative → คูณด้วย -1

In [ ]:
# TODO 1: LOOCV บน Auto Dataset
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# 1. LOOCV สำหรับ degree 1–5
loocv = LeaveOneOut()
loocv_mse = []

for deg in range(1, 6):
    # สร้าง pipeline: PolynomialFeatures → LinearRegression
    # pipe = Pipeline([...])
    # scores = cross_val_score(pipe, X_auto, y_auto, cv=loocv, scoring=...)
    # loocv_mse.append(...)
    pass

# 3. Plot


# 4. Optimal degree
# print(f'Optimal degree = {optimal_degree}, LOOCV MSE = {min_mse:.4f}')


## Part 2: k-Fold Cross-Validation

**Part นี้เราจะใช้ k-Fold CV** ซึ่งเป็น practical trade-off ระหว่าง LOOCV (bias ต่ำ) กับ Validation Set (variance สูง) k-Fold CV แบ่งข้อมูลออกเป็น K folds เท่าๆ กัน แล้ว train บน K-1 folds และ validate บน fold ที่เหลือ ทำซ้ำ K ครั้ง K=5 หรือ K=10 เป็น default ที่ใช้กันมากที่สุดใน practice

In [ ]:
# ─── 5-Fold และ 10-Fold CV เปรียบเทียบกับ LOOCV ───────────────────────────────
# วัตถุประสงค์: แสดงว่า k-Fold CV ให้ผลใกล้เคียง LOOCV แต่เร็วกว่ามาก

degrees = range(1, 8)
mse_results = {'LOOCV': [], '10-Fold CV': [], '5-Fold CV': []}

for deg in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr', LinearRegression())
    ])
    
    # LOOCV (ใช้ n folds)
    scores_loocv = cross_val_score(pipe, X_auto, y_auto,
                                   cv=LeaveOneOut(),
                                   scoring='neg_mean_squared_error')
    mse_results['LOOCV'].append(-scores_loocv.mean())
    
    # 10-Fold CV
    scores_10 = cross_val_score(pipe, X_auto, y_auto,
                                cv=KFold(n_splits=10, shuffle=True, random_state=42),
                                scoring='neg_mean_squared_error')
    mse_results['10-Fold CV'].append(-scores_10.mean())
    
    # 5-Fold CV
    scores_5 = cross_val_score(pipe, X_auto, y_auto,
                               cv=KFold(n_splits=5, shuffle=True, random_state=42),
                               scoring='neg_mean_squared_error')
    mse_results['5-Fold CV'].append(-scores_5.mean())

# Plot
plt.figure(figsize=(10, 5))
colors = {'LOOCV': 'steelblue', '10-Fold CV': 'tomato', '5-Fold CV': 'green'}
for name, mse_list in mse_results.items():
    plt.plot(list(degrees), mse_list, 'o-', lw=2.5, markersize=7,
             color=colors[name], label=name)

plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('CV MSE', fontsize=12)
plt.title('CV MSE vs Polynomial Degree: LOOCV vs 10-Fold vs 5-Fold')
plt.legend()
plt.tight_layout()
plt.show()

# แสดงว่า k-fold ใกล้เคียง LOOCV
for name, mse_list in mse_results.items():
    best_deg = list(degrees)[np.argmin(mse_list)]
    print(f'{name}: optimal degree = {best_deg}, min MSE = {min(mse_list):.4f}')

### 🔰 TODO 2 (Medium): CV สำหรับ KNN Classifier บน Default Dataset

ใน Week 12 เราใช้ KNN Classifier กับ K ที่เลือกเองโดยประมาณ ตอนนี้เราจะใช้ k-Fold CV เลือก K อย่าง **principled** CV ช่วยให้เราเลือก K ที่ minimize test error จริงๆ โดยไม่ต้องใช้ test set ข้อมูลที่ควรใช้ในการ train เท่านั้น

**สิ่งที่ต้องทำ:**
1. โหลด Default dataset (balance + income + student_num → default_num)
2. ใช้ 5-Fold CV (stratified เพราะ imbalanced) หา optimal K สำหรับ KNN
   - ทดสอบ K = 1, 3, 5, 7, 10, 15, 20, 30
   - ใช้ `scoring='f1'` เพราะ imbalanced
3. Plot CV F1 vs K
4. Fit final model ด้วย optimal K บน full train set
5. Report test F1 ที่ optimal K vs K=5 (ที่ใช้ก่อน)

**Hint**: ใช้ `StratifiedKFold` แทน `KFold` สำหรับ imbalanced classification

In [ ]:
# TODO 2: CV สำหรับ KNN Classifier
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# โหลด Default dataset
try:
    default = pd.read_csv('https://www.statlearning.com/s/Default.csv', index_col=0)
    default.columns = default.columns.str.lower()
except:
    np.random.seed(0); n2 = 10000
    balance = np.clip(np.random.exponential(900, n2), 0, 3000)
    income  = np.random.normal(35000, 15000, n2)
    student = np.random.choice([0, 1], n2, p=[0.7, 0.3])
    log_odds = -10.65 + 0.0055*balance - 0.000002*income - 0.65*student
    dy = np.random.binomial(1, expit(log_odds))
    default = pd.DataFrame({'default':['Yes'if d else'No' for d in dy],
                            'student':['Yes'if s else'No' for s in student],
                            'balance':balance,'income':income})
default['default_num'] = (default['default']=='Yes').astype(int)
default['student_num'] = (default['student']=='Yes').astype(int)

X_def = default[['balance','income','student_num']].values
y_def = default['default_num'].values
X_tr, X_te, y_tr, y_te = train_test_split(X_def, y_def, test_size=0.2,
                                            random_state=42, stratify=y_def)

# StandardScaler ก่อน KNN
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# 1. ทดสอบ K ต่างๆ ด้วย 5-Fold CV
k_values = [1, 3, 5, 7, 10, 15, 20, 30]
# cv_f1_scores = []
# for k in k_values:
#     ...


# 3. Plot CV F1 vs K


# 4+5. Fit final model + compare


## Part 3: Bootstrap

**Part นี้เราจะเรียนรู้ Bootstrap** ซึ่งเป็นเทคนิค resampling ที่ powerful ที่สุด แนวคิดคือ simulate new datasets โดย sample **with replacement** จาก training data เดิม ทำซ้ำ B ครั้ง (B=1000) แล้วดู distribution ของ statistic ที่สนใจ Bootstrap ใช้ประเมิน Standard Error ของ statistic ใดๆ — แม้แต่ statistics ที่ไม่มีสูตรปิด เช่น median, correlation coefficient หรือ portfolio weight

In [ ]:
# ─── Bootstrap: ประเมิน SE ของ β̂₁ ใน Simple Linear Regression ───────────────
# วัตถุประสงค์: เปรียบเทียบ SE จาก Bootstrap กับ SE จาก analytical formula
# แสดงว่า Bootstrap ให้ผลใกล้เคียงกับ theory

# สร้าง Simple Linear dataset
np.random.seed(42)
n_slr = 100
x_slr = np.random.uniform(0, 10, n_slr)
y_slr = 2.0 + 1.5 * x_slr + np.random.normal(0, 2, n_slr)  # true β₀=2, β₁=1.5

# Analytical SE ด้วย statsmodels
df_slr = pd.DataFrame({'x': x_slr, 'y': y_slr})
model_sm = smf.ols('y ~ x', data=df_slr).fit()
se_analytical = model_sm.bse['x']
print(f'Analytical SE(β̂₁) = {se_analytical:.4f}')

# Bootstrap SE
B = 1000  # จำนวน bootstrap samples
boot_beta1 = np.zeros(B)

for b in range(B):
    # Sample with replacement
    idx = np.random.choice(n_slr, n_slr, replace=True)  # ← key: replace=True
    x_boot = x_slr[idx]
    y_boot = y_slr[idx]
    
    # Fit OLS บน bootstrap sample
    lr_boot = LinearRegression()
    lr_boot.fit(x_boot.reshape(-1, 1), y_boot)
    boot_beta1[b] = lr_boot.coef_[0]

# Bootstrap SE
se_bootstrap = boot_beta1.std()
print(f'Bootstrap SE(β̂₁) = {se_bootstrap:.4f}  (B={B} bootstrap samples)')
print(f'Difference: {abs(se_analytical - se_bootstrap):.4f}')

# Bootstrap 95% CI
ci_low  = np.percentile(boot_beta1, 2.5)
ci_high = np.percentile(boot_beta1, 97.5)
print(f'Bootstrap 95% CI: [{ci_low:.4f}, {ci_high:.4f}]')

# Plot distribution of bootstrap estimates
plt.figure(figsize=(9, 4))
plt.hist(boot_beta1, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(model_sm.params['x'], color='red', lw=2, label=f'β̂₁ = {model_sm.params["x"]:.3f}')
plt.axvline(ci_low,  color='orange', lw=2, linestyle='--', label='95% CI')
plt.axvline(ci_high, color='orange', lw=2, linestyle='--')
plt.xlabel('Bootstrap β̂₁'); plt.ylabel('Count')
plt.title(f'Bootstrap Distribution of β̂₁ (B={B})\nSE={se_bootstrap:.4f} vs Analytical SE={se_analytical:.4f}')
plt.legend()
plt.tight_layout()
plt.show()

### 🔰 TODO 3 (Medium): Bootstrap สำหรับ Investment Portfolio (ISLP Example)

Bootstrap powerful มากเพราะใช้ได้กับ statistic ใดๆ แม้แต่ไม่มีสูตรปิด ตัวอย่างใน ISLP 5.2 คือการหา **optimal portfolio weight α** ที่ minimize variance ของ portfolio ที่ผสม investment X และ Y:

$$\alpha^* = \frac{\sigma_Y^2 - \sigma_{XY}}{\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}}$$

แต่ σ²ₓ, σ²ᵧ, σ_{XY} ไม่ทราบ → ต้อง estimate จาก data → SE ของ α̂ หาจาก Bootstrap!

**สิ่งที่ต้องทำ:**
1. สร้าง function `alpha_hat(X, Y)` ที่คำนวณ α̂ จากสูตรข้างต้น
2. สร้าง portfolio returns data (code ให้แล้ว)
3. คำนวณ α̂ จาก full data
4. Bootstrap B=1000 ครั้ง: resample, คำนวณ α̂ แต่ละ bootstrap sample
5. คำนวณ Bootstrap SE(α̂) และ 95% CI
6. Plot distribution ของ α̂ bootstrap

In [ ]:
# TODO 3: Bootstrap Portfolio Weight
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# สร้าง portfolio returns data
np.random.seed(1)
n_port = 100
# Returns: X และ Y มี correlation
cov_true = np.array([[1.0, 0.5], [0.5, 1.5]])
returns  = np.random.multivariate_normal([0, 0], cov_true, n_port)
X_port   = returns[:, 0]  # investment X returns
Y_port   = returns[:, 1]  # investment Y returns

# True alpha (from population parameters)
alpha_true = (cov_true[1,1] - cov_true[0,1]) / (cov_true[0,0] + cov_true[1,1] - 2*cov_true[0,1])
print(f'True α* = {alpha_true:.4f}')

# 1. Function alpha_hat
def alpha_hat(X, Y):
    """คำนวณ optimal portfolio weight α"""
    # var_X = np.var(X); var_Y = np.var(Y); cov_XY = np.cov(X, Y)[0,1]
    # return ...
    pass

# 3. คำนวณ α̂ จาก full data


# 4. Bootstrap


# 5+6. SE, CI, plot


## Part 4: CV สำหรับ Model Selection

**Part นี้เราจะรวม CV เข้ากับ model selection pipeline ที่สมบูรณ์** เป้าหมายคือเลือก polynomial degree ที่เหมาะสมสำหรับ Auto dataset และเลือก K สำหรับ KNN โดยใช้ One-Standard-Error Rule ซึ่งบอกว่า: ถ้ามี model ที่ง่ายกว่า (less complex) ที่ CV error อยู่ภายใน 1 SE ของ best model → เลือก simple model เพราะ interpretable กว่าและ overfit น้อยกว่า

### 🔰 TODO 4 (Hard): Full Model Selection Pipeline

ใน TODO สุดท้ายนี้คุณจะสร้าง **full model selection pipeline** ที่รวมทุก concept จาก Week 6–14 เข้าด้วยกัน เราจะเปรียบเทียบ polynomial regression degree 1–8 กับ KNN degree ต่างๆ บน Auto dataset และใช้ **One-Standard-Error Rule** เลือก model ที่ simple ที่สุดที่ยังมี CV error ดีพอ

**สิ่งที่ต้องทำ:**
1. สำหรับ polynomial degree 1–8: คำนวณ 10-Fold CV MSE **พร้อม SE** (std ของ fold scores)
2. สำหรับ KNN K=1–30: คำนวณ 10-Fold CV MSE พร้อม SE
3. Plot: CV MSE ± 1 SE สำหรับทั้ง polynomial และ KNN
4. **One-SE Rule**: หา simplest model ที่ CV MSE ≤ (best MSE + 1 SE)
5. Report: best degree/K และ one-SE-rule degree/K
6. สรุป: degree/K ที่เลือกด้วย one-SE rule แตกต่างจาก best CV score ไหม? เพราะอะไร?

**Hint**: `cross_validate(..., return_train_score=True)` → `'test_score'` คือ CV score แต่ละ fold

In [ ]:
# TODO 4: Full Model Selection Pipeline
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

kf_10 = KFold(n_splits=10, shuffle=True, random_state=42)

# 1. Polynomial degrees 1–8
degrees = range(1, 9)
# poly_cv_mean = []; poly_cv_se = []
# for deg in degrees:
#     ...


# 2. KNN K=1–30
k_range = range(1, 31)
# knn_cv_mean = []; knn_cv_se = []
# for k in k_range:
#     ...


# 3. Plot with error bars


# 4. One-SE Rule


# 5+6. สรุป


## Case Study: CV ใน Industry — Model Selection for Credit Scoring

**Scenario**  
ทีม ML ของธนาคารต้องเลือกระหว่าง Logistic Regression degree=1, Polynomial Logistic degree=2, LDA, QDA และ KNN สำหรับ credit default prediction ด้วย n=5,000 records team ต้องการ method ที่ให้ test AUC สูงสุดแต่ไม่ overfit

**Data**: Default dataset, 5,000 obs, features: balance, income, student

**Method — 5-Fold Stratified CV:**  
ใช้ CV AUC เปรียบเทียบ 5 models, ทำ 5 รอบ CV (different random seeds) เพื่อ estimate variance

**Result**  
| Model | CV AUC Mean | CV AUC SE |
|-------|------------|----------|
| Logistic (degree=1) | 0.948 | 0.003 |
| LDA | 0.946 | 0.004 |
| KNN (K=10, CV optimal) | 0.921 | 0.008 |

**Insight**  
Logistic Regression ดีที่สุดด้วย AUC=0.948 และ SE ต่ำสุด ควรเลือก Logistic Regression เพราะ (1) AUC ดีที่สุด (2) interpretable (3) SE ต่ำ = stable

## สรุปสิ่งที่เรียนรู้

| Method | เมื่อใช้ | Python |
|--------|---------|--------|
| Validation Set | ง่ายสุด แต่ variance สูง | `train_test_split()` |
| LOOCV | n เล็ก, low bias | `LeaveOneOut()` |
| k-Fold CV (k=5,10) | practical default | `KFold(n_splits=10)` |
| Stratified k-Fold | imbalanced classification | `StratifiedKFold()` |
| Bootstrap | SE ของ statistic ใดๆ | manual loop + `np.random.choice(..., replace=True)` |

## คำถาม Reflection

1. **ทำไม k=10 fold CV ถึงดีกว่า k=5 fold CV ในแง่ Bias-Variance?** Training set ใน k=10 มีขนาดเท่าไร (เป็น % ของ total data)? เปรียบเทียบกับ k=5?

2. **Bootstrap กับ k-Fold CV แตกต่างกันอย่างไร?** ใช้คำตอบจาก Lab นี้ประกอบ